<a href="https://colab.research.google.com/github/cedo-cloud/plp-da-a1-gikomba-stall-analysis/blob/main/stall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wanjiku's Fresh-Produce Stall Analysis
## Two-Week Sales and Stock Performance Report

This notebook analyses 14 days of sales and current stock
to assess business performance, identify sales patterns,
and recommend products for restocking.

In [8]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100, 2500, 2200, 3100, 2400, 3900, 5600, 4700] # 14 days, KES

days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2



# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg

stock = {

"Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},

"Onions":  {"qty": 40, "reorder": 20, "cost": 70, "price": 100},

"Sukuma":  {"qty": 9, "reorder": 15, "cost": 30, "price": 50},

"Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},

"Avocado": {"qty": 12, "reorder": 10, "cost": 80, "price": 140},

}

## Task 1: Overall Sales Metrics & Best/Worst Day Identification
**Approach:** Built-in core Python functions (`sum()`, `len()`, `max()`, `min()`) are used to calculate aggregate sales. `list.index()` determines the zero-based index of peak and low sales to infer the corresponding day name and week number.

In [9]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)

highest_sales = max(daily_sales)
lowest_sales = min(daily_sales)

best_index = daily_sales.index(highest_sales)
worst_index = daily_sales.index(lowest_sales)

best_day = days[best_index]
worst_day = days[worst_index]

best_week = best_index // 7 + 1
worst_week = worst_index // 7 + 1

print("Total Sales:", total_sales)
print("Average Sales:", round(average_sales, 2))
print("Highest Sales:", highest_sales, best_day, "Week", best_week)
print("Lowest Sales:", lowest_sales, worst_day, "Week", worst_week)

Total Sales: 46450
Average Sales: 3317.86
Highest Sales: 5600 Sat Week 2
Lowest Sales: 1950 Tue Week 1


## Task 2: Week-on-Week Growth Analysis
**Approach:** Defined `week_totals()` using list slicing to separate Week 1 (indices 0–6) and Week 2 (indices 7–13). Percentage growth is calculated as `((Week 2 - Week 1) / Week 1) * 100`.

In [10]:
def week_totals(sales):
  #Return the total sales for week 1 and week 2.
  week1 = sum(sales[:7])
  week2 = sum(sales[7:])
  return week1, week2

week1_total, week2_total = week_totals(daily_sales)

percentage_growth = ((week2_total - week1_total) / week1_total) * 100

print(f"Week 1 total: KES{week1_total:,}")
print(f"Week 2 total: KES{week2_total:,}")
print(f"Week-on-Week Growth: {percentage_growth:.2f}%")




Week 1 total: KES22,050
Week 2 total: KES24,400
Week-on-Week Growth: 10.66%


## Task 3: Daily Sales Performance Classification
**Approach:** Created `classify_day()` to evaluate sales amounts against thresholds (+20% for "Strong", -20% for "Weak"). Iterated through all days and tracked counts using a dictionary accumulator.

In [12]:
def classify_day(amount, average):
  if amount > average * 1.20:
    return "Strong"
  elif amount < average * 0.80:
    return "Weak"
  else:
    return "Normal"

categories = {"Strong":0, "Normal":0, "Weak":0}

for sales in daily_sales:
  category = classify_day(sales, average_sales)
  categories[category] += 1

print("daily sales Classification: ")
for category, count in categories.items():
  print(f"{category}: {count} days")

daily sales Classification: 
Strong: 4 days
Normal: 4 days
Weak: 6 days


## Task 4: Weekday vs. Weekend Sales Comparison
**Approach:** Categorized sales by day type based on position in the week cycle (indices 0–4 for weekdays, 5–6 for weekends). Calculated averages and determined the percentage difference relative to weekdays.

In [16]:
weekday_sales = daily_sales[:5] + daily_sales[7:12]
weekend_sales = daily_sales[5:7] + daily_sales[12:14]

weekday_average = sum(weekday_sales) / len(weekday_sales)
weekend_average = sum(weekend_sales) / len(weekend_sales)

difference_percentage = (
    (weekend_average - weekday_average) / weekday_average) * 100

print("Weekday Average:", round(weekday_average, 2))
print("Weekend Average:", round(weekend_average, 2))
print("Weekend Difference:", round(difference_percentage, 2),"%")

Weekday Average: 2685.0
Weekend Average: 4900.0
Weekend Difference: 82.5 %


## Task 5: Stock Reorder Assessment & Gross Margin Calculation
**Approach:** Defined `products_to_reorder()` to iterate over the `stock` dictionary using `.items()`. Identified items where `qty <= reorder` and calculated gross profit margin `(price - cost) / price` for all items.

In [34]:
def products_to_reorder(stock):
    """Return products at or below their reorder levels."""
    reorder_list = []

    for product, details in stock.items():
        if details["qty"] <= details["reorder"]:
            reorder_list.append(product)

    return reorder_list


reorder_list = products_to_reorder(stock)

print("Products to reorder:", reorder_list, "\n")

print(f"{'Product':<12} {'Cost/kg':>13} {'Selling price/kg':>23} {'Gross margin':>14}")
print("-" * 67)

for product, details in stock.items():
    cost = details["cost"]
    price = details["price"]

    margin = ((price - cost) / price) * 100

    print(
        f"{product:<12} "
        f"{'KES ' + format(cost, ','):>12} "
        f"{'KES ' + format(price, ','):>20} "
        f"{margin:>13.2f}%"
    )

Products to reorder: ['Tomatoes', 'Sukuma'] 

Product            Cost/kg        Selling price/kg   Gross margin
-------------------------------------------------------------------
Tomatoes           KES 60               KES 90         33.33%
Onions             KES 70              KES 100         30.00%
Sukuma             KES 30               KES 50         40.00%
Potatoes           KES 50               KES 75         33.33%
Avocado            KES 80              KES 140         42.86%


## Task 6: Stall Performance Report Generation
**Approach:** Formatted all calculated figures into a clean text-based report using f-strings and thousands separators (`:,`).

In [36]:
print("=" * 50)
print("          STALL PERFORMANCE REPORT")
print("=" * 50)

print("\nSALES SUMMARY")
print(f"Total sales:          KES {total_sales:,}")
print(f"Average daily sales:  KES {average_sales:,.2f}")
print(f"Highest sales:        KES {highest_sales:,}")
print(f"Best day:             {best_day}, Week {best_week}")
print(f"Lowest sales:         KES {lowest_sales:,}")
print(f"Worst day:            {worst_day}, Week {worst_week}")

print("\nWEEKLY PERFORMANCE")
print(f"Week 1 total:         KES {week1_total:,}")
print(f"Week 2 total:         KES {week2_total:,}")
print(f"Sales growth:         {percentage_growth:.2f}%")

print("\nDAILY CLASSIFICATION")
for category, count in categories.items():
    print(f"{category:<10} {count:>3} days")

print("\nDAY-TYPE COMPARISON")
print(f"Weekday average:      KES {weekday_average:,.2f}")
print(f"Weekend average:      KES {weekend_average:,.2f}")
print(f"Weekend difference:   {difference_percentage:.2f}%")

print("\nSTOCK AND GROSS MARGINS")
for product, details in stock.items():
    margin = (
        (details["price"] - details["cost"])
        / details["price"]
    ) * 100

    status = "REORDER" if product in reorder_list else "Sufficient"

    print(
        f"{product:<12} "
        f"Qty: {details['qty']:>3} kg | "
        f"Margin: {margin:>5.2f}% | "
        f"{status}"
    )

print("=" * 50)

          STALL PERFORMANCE REPORT

SALES SUMMARY
Total sales:          KES 46,450
Average daily sales:  KES 3,317.86
Highest sales:        KES 5,600
Best day:             Sat, Week 2
Lowest sales:         KES 1,950
Worst day:            Tue, Week 1

WEEKLY PERFORMANCE
Week 1 total:         KES 22,050
Week 2 total:         KES 24,400
Sales growth:         10.66%

DAILY CLASSIFICATION
Strong       4 days
Normal       4 days
Weak         6 days

DAY-TYPE COMPARISON
Weekday average:      KES 2,685.00
Weekend average:      KES 4,900.00
Weekend difference:   82.50%

STOCK AND GROSS MARGINS
Tomatoes     Qty:  18 kg | Margin: 33.33% | REORDER
Onions       Qty:  40 kg | Margin: 30.00% | Sufficient
Sukuma       Qty:   9 kg | Margin: 40.00% | REORDER
Potatoes     Qty:  55 kg | Margin: 33.33% | Sufficient
Avocado      Qty:  12 kg | Margin: 42.86% | Sufficient


## Task 7: Recommendations for Wanjiku

1. **Immediately Restock Tomatoes and Sukuma:** Inventory for Tomatoes (18 kg) and Sukuma (9 kg) has dropped below their respective reorder thresholds of 25 kg and 15 kg. Wanjiku should restock these products promptly to reduce the risk of stockouts and lost sales.
2. **Capitalize on Weekend Traffic Dynamics:** Weekend sales average **KES 4,900**, representing an **82.50% increase** over the weekday average of **KES 2,685**. Wanjiku should prepare additional stock by Friday afternoon and consider arranging extra help on Saturdays and Sundays if customer demand requires it.
3. **Prioritize High-Margin Inventory:** Avocado yields the highest gross profit margin at **42.86%** (KES 60 profit/kg), whereas Onions yield the lowest at **30.00%**. Wanjiku should monitor demand and spoilage before increasing avocado stock and consider promotions or bundles for onions to improve sales without unnecessarily reducing profit.